# DQN Hyperparameter Experiments — Miracle

10 experiments (batch size + exploration schedule) and saves results to CSV.



## Cell 1 — Install dependencies

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" "ale-py"

## Cell 2 — Training code (unchanged)

In [ ]:
import os
import json
import argparse
import ale_py
import gymnasium as gym

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import ( EvalCallback, CheckpointCallback, CallbackList,)


# Argument Parser
def parse_args():
    parser = argparse.ArgumentParser(description="Train a DQN agent on an Atari environment.")

    parser.add_argument("--experiment-name", type=str, default="baseline")
    parser.add_argument("--env", type=str, default="ALE/Pong-v5")
    parser.add_argument("--policy", type=str, default="CnnPolicy", choices=["CnnPolicy", "MlpPolicy"])

    parser.add_argument("--timesteps", type=int, default=500_000)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--buffer-size", type=int, default=100_000)
    parser.add_argument("--learning-starts", type=int, default=20_000)

    parser.add_argument("--train-freq", type=int, default=4)
    parser.add_argument("--target-update", type=int, default=10_000)

    parser.add_argument("--eps-start", type=float, default=1.0)
    parser.add_argument("--eps-end", type=float, default=0.05)
    parser.add_argument("--eps-fraction", type=float, default=0.10)

    parser.add_argument("--eval-freq", type=int, default=10_000)
    parser.add_argument("--checkpoint-freq", type=int, default=100_000)

    parser.add_argument("--verbose", type=int, default=0)

    return parser.parse_args()


# Environment

def to_ram_env_id(env_name: str) -> str:
    if "-ram" in env_name:
        return env_name
    if env_name.endswith("-v5"):
        return env_name[: -len("-v5")] + "-ram-v5"
    return env_name.replace("-v", "-ram-v")


def create_env(env_name, seed, policy):
    if policy == "CnnPolicy":
        env = make_atari_env( env_name, n_envs=1, seed=seed, )
        env = VecFrameStack(  env,  n_stack=4, )
    else:
        ram_env_name = to_ram_env_id(env_name)
        env = make_atari_env(  ram_env_name, n_envs=1, seed=seed, )

    return env


# DQN Model
def create_model(env, args, log_dir):

    model = DQN(
        policy=args.policy,
        env=env,

        learning_rate=args.lr,
        gamma=args.gamma,

        batch_size=args.batch_size,
        buffer_size=args.buffer_size,
        learning_starts=args.learning_starts,

        train_freq=args.train_freq,
        target_update_interval=args.target_update,

        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_fraction,

        verbose=args.verbose,

        tensorboard_log=log_dir,
    )

    return model


# Custom Evaluation Callback
class CompactEvalCallback(EvalCallback):
    def __init__(self, *args, csv_file=None, **kwargs):
        super().__init__(*args, **kwargs)

        self.csv_file = csv_file

        # Create CSV and write header
        if self.csv_file is not None:
            with open(self.csv_file, "w") as f:
                f.write("timesteps,reward,best_reward,exploration_rate\n")

    def _on_step(self) -> bool:
        continue_training = super()._on_step()

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:

            print(
                f"[{self.num_timesteps:>7,}] "
                f"Reward: {self.last_mean_reward:>7.2f} | "
                f"Best: {self.best_mean_reward:>7.2f} | "
                f"Exploration: {self.model.exploration_rate:.3f}"
            )

            # Save results to CSV
            if self.csv_file is not None:
                with open(self.csv_file, "a") as f:
                    f.write(
                        f"{self.num_timesteps},"
                        f"{self.last_mean_reward},"
                        f"{self.best_mean_reward},"
                        f"{self.model.exploration_rate}\n"
                    )

        return continue_training

# Training
def train(args):
    base_dir = "/content/drive/MyDrive/"

    assert os.path.isdir("/content/drive/MyDrive"), (
        "Google Drive doesn't appear to be mounted. "
        "Run drive.mount('/content/drive') in a cell before training."
    )
    model_dir = os.path.join(base_dir, "models", args.experiment_name)
    log_dir = os.path.join( base_dir, "logs", args.experiment_name, )

    checkpoint_dir = os.path.join(  base_dir, "checkpoints", args.experiment_name, )

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)

    with open(
        os.path.join(model_dir, "config.json"),
        "w",
    ) as f:

        json.dump(
            vars(args),
            f,
            indent=4,
        )

    env = create_env( args.env, args.seed, args.policy, )

    eval_env = create_env( args.env, args.seed + 100, args.policy, )

    model = create_model( env, args, log_dir,)

    checkpoint_callback = CheckpointCallback(
        save_freq=args.checkpoint_freq,
        save_path=checkpoint_dir,
        name_prefix="dqn_checkpoint",
    )

    eval_callback = CompactEvalCallback(
        eval_env,
        best_model_save_path=model_dir,
        log_path=log_dir,
        eval_freq=args.eval_freq,
        deterministic=True,
        render=False,
        verbose=0,
        n_eval_episodes=10,
    )

    callbacks = CallbackList( [checkpoint_callback, eval_callback,])

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=callbacks,
            log_interval=10,
        )

        model.save( os.path.join( model_dir, "dqn_model", ) )

        print("\nTraining completed successfully!")
        print(f"Final model    : {os.path.join(model_dir, 'dqn_model.zip')}")
        print(f"Best model     : {os.path.join(model_dir, 'best_model.zip')}")
        print(f"Config         : {os.path.join(model_dir, 'config.json')}")

    finally:
        env.close()
        eval_env.close()


# Main

def main():
    args = parse_args()
    train(args)


# (main() intentionally not called here)

## Cell 3 — Run the 10 experiments

Results append to CSV after each run, so an interruption never loses completed
work. Re-running skips whatever already finished.

In [ ]:
import os, csv, time, types
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUTPUT_DIR, "hyperparameter_results.csv")

TIMESTEPS = 100_000

BASELINE = dict(
    env="ALE/Pong-v5", policy="CnnPolicy",
    lr=1e-4, gamma=0.99, batch_size=32,
    buffer_size=100_000, learning_starts=10_000,
    train_freq=4, target_update=10_000,
    eps_start=1.0, eps_end=0.05, eps_fraction=0.10,
    seed=42, verbose=0,
)

EXPERIMENTS = [
    {"name": "Exp21", "batch_size": 16},
    {"name": "Exp22", "batch_size": 64},
    {"name": "Exp23", "batch_size": 128},
    {"name": "Exp24", "batch_size": 256},
    {"name": "Exp25", "eps_end": 0.01},
    {"name": "Exp26", "eps_end": 0.10},
    {"name": "Exp27", "eps_fraction": 0.05},
    {"name": "Exp28", "eps_fraction": 0.20},
    {"name": "Exp29", "eps_start": 0.90},
    {"name": "Exp30", "eps_start": 0.80},
]

FIELDS = ["name", "policy", "lr", "gamma", "batch_size", "eps_start",
          "eps_end", "eps_fraction", "timesteps", "mean_reward",
          "std_reward", "ep_len_mean", "minutes"]


class Progress(BaseCallback):
    """Logs reward trend and episode length (both required by the brief)."""
    def __init__(self, every=25_000):
        super().__init__()
        self.every = every
        self.last_len = None

    def _on_step(self):
        if self.n_calls % self.every == 0 and len(self.model.ep_info_buffer) > 0:
            rew = float(np.mean([e["r"] for e in self.model.ep_info_buffer]))
            ln = float(np.mean([e["l"] for e in self.model.ep_info_buffer]))
            self.last_len = ln
            print(f"    step {self.num_timesteps:>7,} | ep_rew_mean {rew:6.1f} "
                  f"| ep_len_mean {ln:6.0f}", flush=True)
        return True


def append_row(row):
    exists = os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if not exists:
            w.writeheader()
        w.writerow(row)


done = set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV) as f:
        done = {r["name"] for r in csv.DictReader(f)}

for cfg in EXPERIMENTS:
    if cfg["name"] in done:
        print(f"Skipping {cfg['name']} (already done)")
        continue

    overrides = {k: v for k, v in cfg.items() if k != "name"}
    args = types.SimpleNamespace(**{**BASELINE, **overrides})

    print("\n" + "=" * 60)
    print(f"{cfg['name']}: {overrides}")
    start = time.time()

    env = create_env(args.env, args.seed, args.policy)
    eval_env = create_env(args.env, args.seed + 100, args.policy)
    model = create_model(env, args, None)

    progress = Progress()
    model.learn(total_timesteps=TIMESTEPS, callback=progress, log_interval=10**9)

    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    minutes = (time.time() - start) / 60

    env.close()
    eval_env.close()

    append_row({
        "name": cfg["name"], "policy": args.policy, "lr": args.lr,
        "gamma": args.gamma, "batch_size": args.batch_size,
        "eps_start": args.eps_start, "eps_end": args.eps_end,
        "eps_fraction": args.eps_fraction, "timesteps": TIMESTEPS,
        "mean_reward": round(float(mean_r), 2),
        "std_reward": round(float(std_r), 2),
        "ep_len_mean": round(progress.last_len) if progress.last_len else "",
        "minutes": round(minutes, 1),
    })

    print(f"{cfg['name']} done in {minutes:.1f} min -> "
          f"mean reward {mean_r:.1f} +/- {std_r:.1f}")

print(f"\nAll done. Results saved to {RESULTS_CSV}")

## Cell 4 — Print the table for the README

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
try:
    display(df)
except NameError:
    print(df.to_string(index=False))

print("\n--- Paste this into the README ---\n")
print("**MEMBER NAME:** Miracle\n")
print("| # | Hyperparameter Set | Mean Reward | Ep Length | Noted Behavior |")
print("|---|---|---|---|---|")
for _, r in df.iterrows():
    hp = (f"lr={r['lr']}, gamma={r['gamma']}, batch={int(r['batch_size'])}, "
          f"eps_start={r['eps_start']}, eps_end={r['eps_end']}, "
          f"eps_decay={r['eps_fraction']}")
    print(f"| {r['name']} | {hp} | {r['mean_reward']} | "
          f"{r['ep_len_mean']} | _(fill in)_ |")

## Cell 5 — MLP vs CNN comparison

Task 1 requires comparing the two policy types. This runs both at the same short
budget so the difference is attributable to architecture alone.

Note: this uses `MlpPolicy` on the same pixel observations as `CnnPolicy`. That
is the comparison the brief asks for — an MLP flattens the image and loses all
spatial structure, which is exactly why it fails here.

In [ ]:
policy_results = {}

for pol in ["CnnPolicy", "MlpPolicy"]:
    print("\n" + "=" * 60)
    print(f"Policy: {pol}")

    args = types.SimpleNamespace(**{**BASELINE, "policy": pol,
                                    "learning_starts": 5_000})

    # Build the pixel env for BOTH policies so only the architecture differs.
    from stable_baselines3.common.env_util import make_atari_env
    from stable_baselines3.common.vec_env import VecFrameStack

    env = make_atari_env(args.env, n_envs=1, seed=args.seed)
    env = VecFrameStack(env, n_stack=4)
    eval_env = make_atari_env(args.env, n_envs=1, seed=args.seed + 100)
    eval_env = VecFrameStack(eval_env, n_stack=4)

    model = create_model(env, args, None)
    model.learn(total_timesteps=50_000, callback=Progress(every=10_000),
                log_interval=10**9)

    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    policy_results[pol] = (float(mean_r), float(std_r))
    print(f"{pol}: {mean_r:.1f} +/- {std_r:.1f}")

    env.close()
    eval_env.close()

print("\n--- MLP vs CNN (50,000 steps each) ---")
for pol, (m, s) in policy_results.items():
    print(f"{pol:12s} mean reward {m:6.1f} +/- {s:.1f}")